In [1]:
# Setup: put src/ on the path so the notebook runs from any working directory.
import os, sys
_p = os.getcwd()
while _p != os.path.dirname(_p):
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, os.path.join(_p, "src"))
        break
    _p = os.path.dirname(_p)

#import libraries
import json
import time
import pandas as pd

import numpy as np
import matplotlib.pyplot as plt

import matplotlib.ticker as mticker

from pathlib import Path
import seaborn as sns
from sensitivity_analysis import run_sensitivity_analysis
from config import RESULTS_SEMI_DIR


In [2]:
BASE_DIR = Path(RESULTS_SEMI_DIR)

model = "all"

CONFIGS = [
    ("0.1",          f"{model}_0.1"),
    ("0.15",         f"{model}_0.15"),
    ("0.2",          f"{model}_0.2"),
    ("0.25",         f"{model}_0.25"),
    ("0.3",          f"{model}_0.3"),
]

In [3]:
def load_config(label: str, folder: str) -> dict:
    path = BASE_DIR / folder

    with open(path / 'topic_words.json') as f:
        words = json.load(f)

    info = pd.read_csv(path / 'topic_info.csv')
    doc  = pd.read_csv(path / 'document_topics.csv')

    total_docs   = info['Count'].sum()
    assigned_docs = info[info['Topic'] != -1]['Count'].sum()
    outlier_docs  = info[info['Topic'] == -1]['Count'].values[0]

    return {
        "label":         label,
        "folder":        folder,
        "words":         words,
        "info":          info,
        "doc":           doc,
        "total_docs":    total_docs,
        "assigned_docs": assigned_docs,
        "outlier_docs":  outlier_docs,
    }


In [13]:
configs = {}

for label, folder in CONFIGS:
    try:
        configs[label] = load_config(label, folder)
        n_topics = len(configs[label]['words'])
        print(f"[OK] {label:>14}  ->  {n_topics} topics, "
              f"{configs[label]['outlier_docs']:,} outliers")
    except FileNotFoundError as e:
        print(f"[MISSING] {label}: {e}")

required = ["0.1", "0.15", "0.2", "0.25", "0.3"]
missing = [label for label in required if label not in configs]
if missing:
    raise FileNotFoundError(
        "Missing sensitivity topic outputs for " + ", ".join(missing) +
        ". Run data/synthetic/make_synthetic_data.py for synthetic stand-ins "
        "or generate these configs with the BERTopic fitting pipeline."
    )


[OK]            0.1  →  20 topics, 423,049 outliers
[OK]           0.15  →  29 topics, 440,327 outliers
[OK]            0.2  →  29 topics, 431,423 outliers
[OK]           0.25  →  24 topics, 411,902 outliers
[OK]            0.3  →  29 topics, 421,775 outliers


In [5]:
merged_all = configs["0.1"]["doc"].rename(columns={"assigned_topic": "topic_01"})

for label, col in [("0.15", "topic_015"),
                   ("0.2",  "topic_02"),
                   ("0.25", "topic_025"),
                   ("0.3",  "topic_03"),]:
    df = configs[label]["doc"][["document_index", "assigned_topic"]].rename(columns={"assigned_topic": col})
    merged_all = merged_all.merge(df, on="document_index")

print(merged_all.shape)
print(merged_all.head())

(890540, 8)
   document_index  label  topic_01  assigned_topic_probability  topic_015  \
0               0     15         0                         1.0         -1   
1               1     50         2                         1.0          2   
2               2    254        -1                         0.0         -1   
3               3    308        -1                         0.0         -1   
4               4    266        -1                         0.0         -1   

   topic_02  topic_025  topic_03  
0        -1         -1        -1  
1         3          2         2  
2        -1         -1        -1  
3        -1         -1        -1  
4        -1         -1        -1  


In [13]:
results = run_sensitivity_analysis(
    merged_all = merged_all,   # your existing DataFrame
    configs    = configs,      # your existing configs dict
    output_dir = f"sensitivity_results/{model}/",
)

Output directory : /home/s215005/traffic_project/Thesis-traffic-safety/Our Analysis/draft code/sensitivity_results/all
Documents        : 890,540
Weights          : ['0.1', '0.15', '0.2', '0.25', '0.3']

── Step 1: Aggregate metrics
    w=0.1: 20 topics, 47.5% noise
    w=0.15: 29 topics, 49.4% noise
    w=0.2: 29 topics, 48.4% noise
    w=0.25: 24 topics, 46.3% noise
    w=0.3: 29 topics, 47.4% noise
    [saved] plot_topic_count_noise.png

── Step 2: Document flow
    0.1 → 0.15  (20 → 29 topics)
      stayed 46.5%  changed 44.2%  to_outlier 5.6%  rescued 3.7%
    0.15 → 0.2  (29 → 29 topics)
      stayed 77.3%  changed 13.9%  to_outlier 3.9%  rescued 4.9%
    0.25 → 0.2  (24 → 29 topics)  [reordered]
      stayed 41.6%  changed 40.4%  to_outlier 10.1%  rescued 7.9%
    0.25 → 0.3  (24 → 29 topics)
      stayed 75.9%  changed 7.1%  to_outlier 9.0%  rescued 7.9%
    [saved] plot_flow_stacked.png
    [saved] plot_topic_classification.png

── Step 3: Flow concentration (normalised Shanno